# MIMIC-IV: common-period (2017–2019) comparison and admission-time standardized estimates

Runtime → Run all (about 30–45 minutes; four 500-replicate bootstraps). Downloads `mimic_period_results.xlsx` (aggregate only).

In [ ]:
# ---- Settings ---------------------------------------------------------------
PROJECT  = "ctheg-scores"
SEED     = 20261112
B        = 500     # FINAL run: bootstrap reps for primary estimates
B_SENS   = 200     # FINAL run: reps for sensitivity / 65+ / Oaxaca / by-destination
OUTFILE  = "aim1_results.xlsx"
MIN_CELL = 11      # aggregate counts below this are suppressed in exported tables

import time, warnings
import numpy as np
import pandas as pd
import statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf
import patsy

from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)   # convergence is checked explicitly below

class NotConverged(Exception):
    pass

def check(res, label):
    ok = res.mle_retvals.get("converged", True) if hasattr(res, "mle_retvals") and res.mle_retvals else getattr(res, "converged", True)
    if not ok:
        raise NotConverged(label)
    return res
pd.set_option("display.width", 200, "display.max_columns", 50)
print("statsmodels", statsmodels.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

In [ ]:
# ---- Load the analysis views from BigQuery (no file downloads) ---------------
from google.colab import auth
auth.authenticate_user()          # sign in with the Google account linked to PhysioNet
from google.cloud import bigquery
client = bigquery.Client(project=PROJECT)

COLS = """ins_cat, dc_group, post_icu_days, any_post_icu_stay, dc_ama, dc_acute_transfer,
          time_valid_strict, age_at_admit, gender, race_cat, sofa_day1, cci_no_age,
          adm_type_cat, invasive_vent, vasopressor, icu_los_days_total, first_careunit,
          anchor_year_group, language_cat"""

raw18 = client.query(f"SELECT {COLS} FROM `{PROJECT}.ins_icu.aim1_primary_18_64`").to_dataframe()
raw65 = client.query(f"SELECT {COLS} FROM `{PROJECT}.ins_icu.aim1_older_65plus`").to_dataframe()
print("18-64:", raw18.shape, "| 65+:", raw65.shape)

In [ ]:
# ---- Variable preparation -----------------------------------------------------
DC_LEVELS = ["home", "home_health", "snf", "rehab", "ltach", "other"]   # code 0 = home (reference)

ICU_MAP = {
    "Cardiac Vascular Intensive Care Unit (CVICU)":       "Cardiac",
    "Coronary Care Unit (CCU)":                           "Cardiac",
    "Medical Intensive Care Unit (MICU)":                 "Medical",
    "Medical/Surgical Intensive Care Unit (MICU/SICU)":   "Mixed",
    "Surgical Intensive Care Unit (SICU)":                "Surgical/Trauma",
    "Trauma SICU (TSICU)":                                "Surgical/Trauma",
    "Neuro Surgical Intensive Care Unit (Neuro SICU)":    "Neuro",
    "Neuro Intermediate":                                 "Neuro",
    "Neuro Stepdown":                                     "Neuro",
}   # anything else (PACU, Medicine, 'Intensive Care Unit (ICU)', small surgical units; ~60 patients) -> 'Mixed'
#     (merged 2026-11-12: a separate tiny group caused non-convergence in bootstrap resamples)

REF = {"race_cat": "White", "adm_type_cat": "Emergency", "icu_type": "Medical",
       "anchor_year_group": "2008 - 2010", "language_cat": "English"}

def prep(raw, exposure_levels):
    d = raw[raw["ins_cat"].isin(exposure_levels)].copy()
    d["icu_type"]    = d["first_careunit"].map(ICU_MAP).fillna("Mixed")
    d["female"]      = (d["gender"] == "F").astype(int)
    d["age_c"]       = (d["age_at_admit"] - 50) / 10            # per 10 years, centred at 50
    d["log_icu_los"] = np.log1p(d["icu_los_days_total"].astype(float))
    d["vent"]        = d["invasive_vent"].astype(int)
    d["pressor"]     = d["vasopressor"].astype(int)
    d["any_stay"]    = d["any_post_icu_stay"].astype(int)
    d["days_total"]  = np.where(d["any_stay"] == 1, d["post_icu_days"].astype(float), 0.0)  # two-part: <=4 h counted as 0
    d["post_icu_days"] = d["post_icu_days"].astype(float)
    d["sofa_day1"]   = d["sofa_day1"].astype(float)
    d["cci_no_age"]  = d["cci_no_age"].astype(float)
    d["facility"]    = d["dc_group"].isin(["snf", "rehab", "ltach"]).astype(int)
    for c in ["dc_ama", "dc_acute_transfer", "time_valid_strict"]:
        d[c] = d[c].astype(bool)
    d["dc_code"] = pd.Categorical(d["dc_group"], categories=DC_LEVELS).codes
    d["ins"] = pd.Categorical(d["ins_cat"], categories=exposure_levels)   # first level = reference
    for c, ref in REF.items():                                            # fixed level order, reference first
        levels = [ref] + sorted(x for x in d[c].dropna().unique() if x != ref)
        d[c] = pd.Categorical(d[c], categories=levels)
    n_missing = d[["sofa_day1", "cci_no_age", "dc_code"]].isna().sum()
    assert (d["dc_code"] >= 0).all(), "unexpected discharge group"
    print(f"n = {len(d):,} | missing:", n_missing.to_dict())
    return d.dropna(subset=["sofa_day1", "cci_no_age"]).reset_index(drop=True)

COV = ("age_c + I(age_c**2) + female + C(race_cat) + sofa_day1 + cci_no_age + C(adm_type_cat)"
       " + vent + pressor + log_icu_los + C(icu_type) + C(anchor_year_group)")
COV_LANG = COV + " + C(language_cat)"
EXPO = "C(ins)"

d18 = prep(raw18, ["Private", "Medicaid"])
d18["medicaid"] = (d18["ins_cat"] == "Medicaid").astype(int)
print(d18["ins"].value_counts())

In [ ]:
# ---- Common-period (2017-2019) comparison and admission-time (M0) standardized estimates ----
# Multinomial logit + marginal standardization (g-computation), patient bootstrap, seed 20261113.
# Output: mimic_period_results.xlsx (aggregate only; counts < 11 suppressed).
import time, numpy as np, pandas as pd, statsmodels.formula.api as smf
SEED2, BOOT = 20261113, 500
DC = DC_LEVELS
rawP = client.query(f"SELECT * FROM `{PROJECT}.ins_icu.aim1_primary_18_64`").to_dataframe()
d = prep(rawP, ["Private", "Medicaid"])
adm = pd.to_datetime(d["admittime"]).dt.normalize(); dis = pd.to_datetime(d["dischtime"]).dt.normalize()
d["los_tx"] = ((dis - adm).dt.days).clip(lower=1).astype(float)
d["medicaid"] = (d["ins_cat"] == "Medicaid").astype(int)
COV_M0 = "age_c + I(age_c**2) + female + C(race_cat) + C(adm_type_cat) + C(anchor_year_group)"
COV_M3 = COV
drop_era = lambda c: c.replace(" + C(anchor_year_group)", "")

def std_probs(dd, cov, start=None):
    res = smf.mnlogit(f"dc_code ~ C(ins) + {cov}", dd).fit(disp=0, maxiter=80, method="newton", start_params=start)
    if not res.mle_retvals.get("converged", True):
        raise RuntimeError("not converged")
    out = {}
    pr = {}
    for lvl in ["Private", "Medicaid"]:
        x = dd.copy(); x["ins"] = pd.Categorical([lvl] * len(x), categories=dd["ins"].cat.categories)
        pr[lvl] = np.asarray(res.predict(x)).mean(axis=0)
    fac = [DC.index(k) for k in ["snf", "rehab", "ltach"]]
    for j, k in enumerate(DC):
        out[f"{k}|Private"], out[f"{k}|Medicaid"] = pr["Private"][j], pr["Medicaid"][j]
        out[f"{k}|diff"], out[f"{k}|ratio"] = pr["Medicaid"][j] - pr["Private"][j], pr["Medicaid"][j] / pr["Private"][j]
    a1, a0 = pr["Medicaid"][fac].sum(), pr["Private"][fac].sum()
    out["facility|Private"], out["facility|Medicaid"], out["facility|diff"], out["facility|ratio"] = a0, a1, a1 - a0, a1 / a0
    return out, res

def run(label, dd, cov):
    t0 = time.time()
    point, res = std_probs(dd, cov)
    start = res.params.values.flatten(order="F")
    rng, bs, fail = np.random.default_rng(SEED2), [], 0
    for b in range(BOOT):
        db = dd.iloc[rng.integers(0, len(dd), len(dd))].reset_index(drop=True)
        try:
            bs.append(std_probs(db, cov, start)[0])
        except Exception:
            fail += 1
    bs = pd.DataFrame(bs)
    rows = []
    for k, v in point.items():
        dest, stat = k.split("|")
        lo, hi = np.nanpercentile(bs[k], [2.5, 97.5])
        rows.append(dict(analysis=label, destination=dest, stat=stat, value=v, lo=lo, hi=hi))
    for y in ["los_tx", "days_total"]:
        r = smf.ols(f"{y} ~ medicaid + {cov}", dd).fit(cov_type="HC1")
        b_, s_ = r.params["medicaid"], r.bse["medicaid"]
        rows.append(dict(analysis=label, destination=y, stat="diff (OLS, HC1)", value=b_, lo=b_ - 1.96 * s_, hi=b_ + 1.96 * s_))
    nM, nP = int(dd["medicaid"].sum()), int((1 - dd["medicaid"]).sum())
    rows.append(dict(analysis=label, destination="n", stat="Medicaid / private", value=nM if nM >= 11 else np.nan, lo=nP if nP >= 11 else np.nan, hi=np.nan))
    rows.append(dict(analysis=label, destination="bootstrap", stat="replicates used / requested", value=len(bs), lo=BOOT, hi=fail))
    print(f"{label}: n={len(dd):,}, {len(bs)}/{BOOT} replicates, {time.time() - t0:.0f}s")
    return pd.DataFrame(rows)

sub = d[d["anchor_year_group"].astype(str) == "2017 - 2019"].reset_index(drop=True)
for c in sub.select_dtypes("category"):
    sub[c] = sub[c].cat.remove_unused_categories()
res_all = pd.concat([
    run("2008-2022, M0 admission-time", d, COV_M0),
    run("2008-2022, M3 paper covariates", d, COV_M3),
    run("2017-2019, M0 admission-time", sub, drop_era(COV_M0)),
    run("2017-2019, M3 paper covariates", sub, drop_era(COV_M3)),
], ignore_index=True)
crude = pd.crosstab(sub["dc_group"], sub["ins_cat"]).reset_index()
for c in crude.columns[1:]:
    crude[c] = crude[c].where(crude[c] >= 11)
meta = pd.DataFrame(dict(item=["seed", "bootstrap", "statsmodels", "pandas", "numpy"],
                         value=[SEED2, BOOT, statsmodels.__version__, pd.__version__, np.__version__]))
with pd.ExcelWriter("mimic_period_results.xlsx") as xw:
    res_all.to_excel(xw, sheet_name="standardized", index=False)
    crude.to_excel(xw, sheet_name="crude_2017_2019", index=False)
    meta.to_excel(xw, sheet_name="meta", index=False)
print(res_all[res_all["stat"].isin(["diff", "diff (OLS, HC1)"])].round(4).to_string(index=False))
from google.colab import files
files.download("mimic_period_results.xlsx")
